# MonolithFarm — Notebook didático com foco em NDVI

Este notebook foi reorganizado para duas finalidades ao mesmo tempo:

1. **entender o projeto de ponta a ponta**;
2. **servir como material de apresentação**.

O foco principal é o **NDVI**. Clima, operação, pragas e solo aparecem aqui como variáveis que ajudam a explicar o comportamento do NDVI ao longo do tempo.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path


OUTPUT_SUBDIR = 'complete_ndvi'
NOTEBOOK_MODE = os.environ.get("MONOLITHFARM_NOTEBOOK_MODE", "auto").strip().lower()
if NOTEBOOK_MODE not in {"auto", "jupyter", "colab"}:
    raise ValueError("MONOLITHFARM_NOTEBOOK_MODE deve ser `auto`, `jupyter` ou `colab`.")

PROFILE_NAME = os.environ.get("MONOLITHFARM_PROFILE", "").strip()
CONFIG_ENV_PATH = os.environ.get("MONOLITHFARM_PATHS_FILE", "").strip()
IN_COLAB_RUNTIME = "google.colab" in sys.modules
USE_COLAB_MODE = NOTEBOOK_MODE == "colab" or (NOTEBOOK_MODE == "auto" and IN_COLAB_RUNTIME)
REQUIRED_PACKAGES = ['duckdb', 'matplotlib', 'numpy', 'pandas', 'pyproj', 'scipy', 'shapely']
COLAB_PROJECT_HINTS = [
    "/content/drive/MyDrive/MonolithFarm",
    "/content/drive/My Drive/MonolithFarm",
    "/content/Projeto-FarmLab",
    "/content/MonolithFarm",
]
COLAB_DATA_HINTS = [f"{project_dir}/data" for project_dir in COLAB_PROJECT_HINTS]


def package_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def first_existing_path(candidates: list[str]) -> Path | None:
    for candidate in candidates:
        path = Path(candidate).expanduser()
        if path.exists():
            return path.resolve()
    return None


def discover_paths_config_file() -> Path | None:
    candidates: list[Path] = []
    if CONFIG_ENV_PATH:
        candidates.append(Path(CONFIG_ENV_PATH).expanduser())

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        candidates.append(candidate / ".monolithfarm.paths.json")
        candidates.append(candidate / "monolithfarm.paths.json")

    seen: set[str] = set()
    for candidate in candidates:
        resolved = candidate.resolve() if candidate.is_absolute() else candidate
        key = str(resolved)
        if key in seen:
            continue
        seen.add(key)
        if resolved.exists():
            return resolved
    return None


def load_paths_config() -> tuple[dict, Path | None]:
    config_path = discover_paths_config_file()
    if config_path is None:
        return {}, None

    payload = json.loads(config_path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError(f"Arquivo de configuracao invalido: {config_path}")
    return payload, config_path


PATHS_CONFIG, PATHS_CONFIG_FILE = load_paths_config()
CONFIG_BASE_DIR = PATHS_CONFIG_FILE.parent.resolve() if PATHS_CONFIG_FILE is not None else None
DEFAULT_PROFILE = str(PATHS_CONFIG.get("default_profile") or "").strip()
ACTIVE_PROFILE = PROFILE_NAME or DEFAULT_PROFILE
PROFILE_SETTINGS = {}
if ACTIVE_PROFILE:
    profiles = PATHS_CONFIG.get("profiles", {})
    if ACTIVE_PROFILE not in profiles:
        raise KeyError(
            f"Perfil `{ACTIVE_PROFILE}` nao encontrado em {PATHS_CONFIG_FILE}."
        )
    profile_value = profiles.get(ACTIVE_PROFILE)
    if isinstance(profile_value, dict):
        PROFILE_SETTINGS = profile_value
elif isinstance(PATHS_CONFIG.get("profile"), dict):
    PROFILE_SETTINGS = PATHS_CONFIG["profile"]


def config_value(key: str):
    return PROFILE_SETTINGS.get(key) if isinstance(PROFILE_SETTINGS, dict) else None


def resolve_config_path(raw_value: object, *, base_dir: Path | None) -> Path | None:
    if raw_value in {None, ""}:
        return None
    path = Path(str(raw_value)).expanduser()
    if not path.is_absolute() and base_dir is not None:
        path = (base_dir / path).resolve()
    else:
        path = path.resolve()
    return path


def mount_colab_drive_if_needed() -> None:
    if not USE_COLAB_MODE:
        return
    if not IN_COLAB_RUNTIME:
        raise RuntimeError(
            "MONOLITHFARM_NOTEBOOK_MODE='colab' foi definido, mas o runtime atual nao e Google Colab."
        )
    from google.colab import drive

    drive_root = Path("/content/drive")
    drive_root.mkdir(parents=True, exist_ok=True)
    if not (drive_root / "MyDrive").exists() and not (drive_root / "My Drive").exists():
        drive.mount("/content/drive")


def find_project_dir() -> Path:
    explicit = os.environ.get("MONOLITHFARM_PROJECT_DIR")
    if explicit:
        return Path(explicit).expanduser().resolve()

    config_project_dir = resolve_config_path(config_value("project_dir"), base_dir=CONFIG_BASE_DIR)
    if config_project_dir is not None:
        return config_project_dir

    if USE_COLAB_MODE:
        hinted_project = first_existing_path(COLAB_PROJECT_HINTS)
        if hinted_project is not None and (hinted_project / "pyproject.toml").exists():
            return hinted_project
        hinted_data = first_existing_path(COLAB_DATA_HINTS)
        if hinted_data is not None and (hinted_data.parent / "pyproject.toml").exists():
            return hinted_data.parent.resolve()

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "Nao foi possivel localizar `pyproject.toml`. Defina MONOLITHFARM_PROJECT_DIR, "
        "MONOLITHFARM_PROFILE ou crie `.monolithfarm.paths.json`."
    )


def find_data_dir(project_dir: Path) -> Path:
    explicit = os.environ.get("MONOLITHFARM_DATA_DIR")
    if explicit:
        return Path(explicit).expanduser().resolve()

    config_data_dir = resolve_config_path(config_value("data_dir"), base_dir=CONFIG_BASE_DIR)
    if config_data_dir is not None:
        return config_data_dir

    if USE_COLAB_MODE:
        hinted_data = first_existing_path(COLAB_DATA_HINTS)
        if hinted_data is not None:
            return hinted_data

    for candidate in [project_dir / "data", project_dir / "FarmLab"]:
        if candidate.exists():
            return candidate.resolve()

    return (project_dir / "data").resolve()


def find_output_dir(project_dir: Path) -> Path:
    explicit = os.environ.get("MONOLITHFARM_OUTPUT_DIR")
    if explicit:
        return Path(explicit).expanduser().resolve()

    config_output_dir = resolve_config_path(config_value("output_dir"), base_dir=CONFIG_BASE_DIR)
    if config_output_dir is not None:
        return config_output_dir

    config_output_root = resolve_config_path(config_value("output_root"), base_dir=CONFIG_BASE_DIR)
    if config_output_root is not None:
        return (config_output_root / OUTPUT_SUBDIR).resolve()

    return (project_dir / OUTPUT_SUBDIR).resolve()


def auto_install_enabled() -> bool:
    explicit = os.environ.get("MONOLITHFARM_AUTO_INSTALL")
    if explicit is not None:
        return explicit == "1"

    config_auto_install = config_value("auto_install")
    if config_auto_install is not None:
        return bool(config_auto_install)

    return USE_COLAB_MODE


mount_colab_drive_if_needed()
PROJECT_DIR = find_project_dir()
DATA_DIR = find_data_dir(PROJECT_DIR)
OUTPUT_DIR = find_output_dir(PROJECT_DIR)
AUTO_INSTALL = auto_install_enabled()

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

if AUTO_INSTALL and any(not package_available(name) for name in REQUIRED_PACKAGES):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR)])
    except Exception:
        subprocess.check_call(["uv", "pip", "install", "--python", sys.executable, "-e", str(PROJECT_DIR)])

print("NOTEBOOK_MODE =", NOTEBOOK_MODE)
print("IN_COLAB_RUNTIME =", IN_COLAB_RUNTIME)
print("USE_COLAB_MODE =", USE_COLAB_MODE)
print("PROFILE_NAME =", PROFILE_NAME or "<default>")
print("PATHS_CONFIG_FILE =", PATHS_CONFIG_FILE)
print("ACTIVE_PROFILE =", ACTIVE_PROFILE or "<none>")
print("PROJECT_DIR =", PROJECT_DIR)
print("DATA_DIR    =", DATA_DIR)
print("OUTPUT_DIR  =", OUTPUT_DIR)
print("AUTO_INSTALL =", AUTO_INSTALL)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Diretorio de dados nao encontrado: {DATA_DIR}")

import ast
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display

from dashboard.lineage.column_catalog import build_feature_catalog, build_workspace_column_catalog
from dashboard.lineage.column_lineage import raw_columns_for_feature, thresholds_for_feature
from dashboard.lineage.docs_registry import DRIVER_DOCUMENTATION, column_documentation_for
from dashboard.lineage.registry import CSV_REGISTRY, FEATURE_REGISTRY, INTERMEDIATE_TABLE_REGISTRY
from farmlab.complete_analysis import build_complete_ndvi_workspace, save_complete_ndvi_outputs
from farmlab.io import discover_dataset_paths, load_ndvi_metadata

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_rows", 2000)

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True

SOURCE_ROOT = PROJECT_DIR
OUTPUTS_DIR = OUTPUT_DIR

workspace = build_complete_ndvi_workspace(DATA_DIR)
save_complete_ndvi_outputs(workspace, OUTPUTS_DIR)
paths = discover_dataset_paths(DATA_DIR)
ndvi_raw = load_ndvi_metadata(paths)

print("PROJECT_DIR =", PROJECT_DIR)
print("DATA_DIR =", DATA_DIR)
print("OUTPUTS_DIR =", OUTPUTS_DIR)
print("Perfil ativo =", os.environ.get("MONOLITHFARM_PROFILE", "<default>"))


## Como este notebook foi organizado

A estrutura foi reorganizada para responder primeiro à pergunta central do projeto:

> **O que aconteceu com o NDVI ao longo do tempo e por que o manejo 4.0 nem sempre ficou acima do convencional?**

Por isso, o fluxo ficou concentrado nas tabelas que mais ajudam a contar essa história:

1. `ndvi_clean` — a base de cenas NDVI válidas;
2. `ndvi_phase_timeline` — a série semanal enriquecida, com fases, eventos e drivers;
3. `pair_weekly_gaps` — a diferença semanal entre 4.0 e convencional;
4. `ndvi_stats_by_area`, `ndvi_outliers`, `pair_effect_tests`, `pair_classic_tests`, `weekly_correlations`;
5. `event_driver_lift`, `final_hypothesis_register` e `decision_summary`.

O objetivo foi **deixar o raciocínio mais claro**, sem mudar a lógica principal do pipeline.


In [ ]:
def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")


def resolve_source_file(file_name: str) -> Path:
    raw_path = Path(file_name)
    candidates = [
        SOURCE_ROOT / raw_path,
        SOURCE_ROOT / "farmlab" / raw_path.name,
        SOURCE_ROOT / "dashboard" / raw_path.name,
        SOURCE_ROOT / "docs" / raw_path.name,
        SOURCE_ROOT / "scripts" / raw_path.name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Arquivo nao encontrado para exibicao de codigo: {file_name}")


def extract_function_source(file_path: Path, function_name: str) -> str:
    source = read_text(file_path)
    module = ast.parse(source)
    lines = source.splitlines()
    for node in ast.walk(module):
        if isinstance(node, ast.FunctionDef) and node.name == function_name:
            start = node.lineno - 1
            end = node.end_lineno
            snippet = lines[start:end]
            numbered = [f"{i + start + 1:04d}: {line}" for i, line in enumerate(snippet)]
            return "\n".join(numbered)
    raise ValueError(f"Funcao {function_name} nao encontrada em {file_path}")


def show_function(file_name: str, function_name: str):
    file_path = resolve_source_file(file_name)
    title = f"### Codigo real — `{function_name}` em `{file_path.relative_to(SOURCE_ROOT)}`"
    display(Markdown(title))
    print(extract_function_source(file_path, function_name))


def load_csv(name: str) -> pd.DataFrame:
    path = OUTPUTS_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"CSV nao encontrado em notebook_outputs/complete_ndvi: {path.name}")
    return pd.read_csv(path)


def show_df(title: str, df: pd.DataFrame, n: int = 5):
    display(Markdown(f"### {title}"))
    display(df.head(n))
    print("shape =", df.shape)


def show_scroll_df(title: str, df: pd.DataFrame, max_height: int = 520):
    display(Markdown(f"### {title}"))
    html = df.to_html(index=False, escape=False)
    display(
        HTML(
            f"<div style='max-height:{max_height}px; overflow:auto; border:1px solid #d0d7de; "
            "border-radius:8px; padding:6px; background:white'>"
            f"{html}</div>"
        )
    )
    print("shape =", df.shape)


def md(text: str):
    display(Markdown(text))



# 1. Origem dos dados

Nesta seção, a ideia é enxergar o `data/` como um **ecossistema de fontes**. Não basta saber que “tem NDVI”; é preciso saber qual pasta representa qual sistema e qual arquivo bruto alimenta cada etapa.


In [ ]:
raw_inventory = []
for path in sorted(DATA_DIR.rglob("*")):
    if not path.is_file():
        continue
    raw_inventory.append(
        {
            "relativo_ao_data": str(path.relative_to(DATA_DIR)),
            "sufixo": path.suffix.lower(),
            "tamanho_kb": round(path.stat().st_size / 1024, 2),
            "pasta_pai": str(path.parent.relative_to(DATA_DIR)),
        }
    )

raw_inventory = pd.DataFrame(raw_inventory)
show_df("Inventario bruto real da pasta data/ (primeiras linhas)", raw_inventory, 30)
print("total_de_arquivos_brutos =", len(raw_inventory))


In [ ]:
def infer_source_group(source_key: str) -> str:
    if source_key.startswith("ndvi_"):
        return "OneSoil"
    if source_key.startswith("weather_"):
        return "Metos"
    if source_key.startswith("traps_") or source_key.startswith("pest_"):
        return "EKOS Pest"
    if source_key.startswith("soil_"):
        return "Cropman"
    return "EKOS Layers / apoio"


source_rows = []
for source_key, value in vars(paths).items():
    if value is None:
        continue
    path = Path(value)
    source_rows.append(
        {
            "fonte": infer_source_group(source_key),
            "source_key": source_key,
            "caminho_completo": str(path),
            "relativo_ao_data": str(path.relative_to(DATA_DIR)) if path.exists() and DATA_DIR in path.parents else str(path),
            "tipo": "diretorio" if path.is_dir() else "arquivo",
            "existe": path.exists(),
            "entra_no_fluxo_ndvi": source_key
            in {
                "ndvi_metadata",
                "weather_hourly",
                "traps_data",
                "traps_events",
                "pest_list",
                "pest_details",
                "soil_analysis",
                "layers_dir",
                "shapes_dir",
            },
        }
    )

source_rows = pd.DataFrame(source_rows).sort_values(["fonte", "source_key"]).reset_index(drop=True)
show_df("Mapa das fontes reais detectadas no ambiente atual", source_rows, 30)



## Arquivos brutos mais importantes para o foco em NDVI

Para esta versão simplificada, os arquivos brutos mais importantes são:

- `ndvi_metadata.csv` → prova quantitativa do NDVI;
- arquivos Metos → chuva, temperatura, umidade e balanço hídrico;
- camadas EKOS → operação/telemetria como explicadores;
- MIIP / armadilhas → pragas como explicadores;
- `soil_analysis.csv` → solo como contexto.


In [ ]:
ndvi_raw_path = Path(paths.ndvi_metadata)
show_df("Bruto principal do NDVI: ndvi_metadata.csv", ndvi_raw, 8)
print("\nArquivo bruto lido =", ndvi_raw_path)
print("\nColunas do bruto NDVI:")
print(ndvi_raw.columns.tolist())



# 2. Dicionário do pipeline

Abaixo está a versão simplificada do pipeline, com foco no que realmente alimenta a história do NDVI.


In [ ]:

pipeline_dict = pd.DataFrame([
    ['ndvi_metadata.csv', 'load_ndvi_metadata', 'io.py', 'CSV bruto do OneSoil', 'DataFrame bruto de cenas NDVI'],
    ['area_metadata (interno)', 'build_area_metadata', 'pairwise.py', 'season_id + mapeamento oficial', 'metadados de área/pareamento'],
    ['weather_daily.csv', 'build_weather_daily', 'pairwise.py', 'Metos horário', 'clima diário'],
    ['weather_weekly (interno)', 'build_weather_weekly', 'pairwise.py', 'weather_daily', 'clima semanal'],
    ['ndvi_clean.csv', 'build_ndvi_clean', 'pairwise.py', 'ndvi bruto + area_metadata + weather_daily', 'base NDVI válida'],
    ['ops_area_daily.csv', 'build_ops_area_daily', 'pairwise.py', 'camadas EKOS', 'operação diária por área'],
    ['miip_daily.csv', 'build_miip_daily', 'pairwise.py', 'traps + pest', 'pragas diárias por área'],
    ['pairwise_weekly_features (interno)', 'build_pairwise_weekly_features', 'pairwise.py', 'NDVI + clima + operação + MIIP', 'base semanal integrada'],
    ['ndvi_phase_timeline.csv', 'build_ndvi_phase_timeline', 'ndvi_deepdive.py', 'pairwise_weekly_features + ops_support_weekly', 'timeline semanal com fases/eventos/drivers'],
    ['ndvi_stats_by_area.csv', 'build_ndvi_stats_by_area', 'complete_analysis.py', 'ndvi_clean', 'estatística descritiva por área'],
    ['pair_weekly_gaps.csv', 'build_pair_weekly_gaps', 'complete_analysis.py', 'ndvi_phase_timeline', 'gaps 4.0 - convencional por semana'],
    ['ndvi_outliers.csv', 'build_ndvi_outliers', 'complete_analysis.py', 'ndvi_clean', 'outliers de NDVI'],
    ['pair_effect_tests.csv', 'build_pair_effect_tests', 'ndvi_crispdm.py', 'ndvi_phase_timeline', 'teste principal das hipóteses de efeito'],
    ['pair_classic_tests.csv', 'build_pair_classic_tests', 'complete_analysis.py', 'pair_weekly_gaps', 'teste estatístico clássico dos gaps'],
    ['weekly_correlations.csv', 'build_weekly_correlations', 'complete_analysis.py', 'transition_model_frame', 'correlações semanais com NDVI'],
    ['event_driver_lift.csv', 'build_event_driver_lift', 'ndvi_crispdm.py', 'ndvi_phase_timeline', 'drivers sobre-representados nas semanas problema'],
    ['final_hypothesis_register.csv', 'build_final_hypothesis_register', 'ndvi_crispdm.py', 'testes + drivers + outlook', 'status final H1-H4'],
    ['decision_summary.csv', 'build_decision_summary', 'ndvi_crispdm.py', 'testes + hipóteses + outlook', 'mensagem final por par'],
], columns=['csv_ou_tabela', 'funcao_geradora', 'arquivo_codigo', 'recebe', 'devolve'])
show_df('Dicionário simplificado do pipeline', pipeline_dict, 30)



# 3. CSV por CSV — foco no que realmente conta para o NDVI

Nesta seção, cada CSV importante é explicado individualmente com:

- nome;
- função que gera;
- código da função;
- entradas;
- filtros aplicados;
- colunas derivadas;
- amostra do resultado;
- onde é usado depois;
- qual hipótese ou gráfico depende dele.


In [ ]:
ndvi_clean = load_csv("ndvi_clean.csv")
weather_daily = load_csv("weather_daily.csv")
weather_weekly = load_csv("weather_weekly.csv")
ops_area_daily = load_csv("ops_area_daily.csv")
miip_daily = load_csv("miip_daily.csv")
pairwise_weekly_features = load_csv("pairwise_weekly_features.csv")
ndvi_phase_timeline = load_csv("ndvi_phase_timeline.csv")
ndvi_stats_by_area = load_csv("ndvi_stats_by_area.csv")
pair_weekly_gaps = load_csv("pair_weekly_gaps.csv")
ndvi_outliers = load_csv("ndvi_outliers.csv")
pair_effect_tests = load_csv("pair_effect_tests.csv")
pair_classic_tests = load_csv("pair_classic_tests.csv")
weekly_correlations = load_csv("weekly_correlations.csv")
event_driver_lift = load_csv("event_driver_lift.csv")
final_hypothesis_register = load_csv("final_hypothesis_register.csv")
decision_summary = load_csv("decision_summary.csv")
area_inventory = load_csv("area_inventory.csv")

csv_inventory = pd.DataFrame(
    {
        "csv": [path.name for path in sorted(OUTPUTS_DIR.glob("*.csv"))],
        "tamanho_kb": [round(path.stat().st_size / 1024, 2) for path in sorted(OUTPUTS_DIR.glob("*.csv"))],
    }
)
show_df("CSVs atualmente disponiveis em notebook_outputs/complete_ndvi", csv_inventory, 40)


## 3.0 Catálogo completo de features, drivers e colunas geradas

Esta seção existe para deixar explícito o que antes ficava espalhado entre código, CSVs e gráficos.

Aqui o notebook documenta:

- **features brutas/derivadas/agregadas/flags/scores**;
- **drivers** usados para explicar semanas-problema do NDVI;
- **colunas dos CSVs finais e intermediários**;
- **origem bruta** de cada feature principal;
- **função geradora**;
- **filtros, agregações e thresholds**;
- **onde a informação aparece depois**;
- **quais hipóteses e gráficos dependem dela**.

Termos importantes:

- `feature`: variável criada ou usada no pipeline.
- `driver`: fator explicativo candidato, por exemplo `solo_exposto` ou `falha_de_dose_na_adubacao`.
- `flag`: feature booleana/indicadora, como `high_soil_flag`.
- `score`: valor sintético, como `risk_flag_count`.
- `raw_columns_resolved`: coluna(s) bruta(s) que alimentam a feature.

**Importante:** driver não significa causa comprovada. Driver significa fator que apareceu associado a semanas problemáticas e que merece investigação.


In [ ]:
feature_catalog = build_feature_catalog()
show_scroll_df(
    "Catálogo de features principais rastreadas",
    feature_catalog[
        [
            "feature",
            "category",
            "definition",
            "born_table",
            "function",
            "source_columns",
            "raw_columns_resolved",
            "raw_sources",
            "transformation",
            "thresholds",
            "filters",
            "appears_in_tables",
            "appears_in_csvs",
            "hypotheses",
            "charts",
        ]
    ],
    max_height=680,
)

missing_raw = feature_catalog[feature_catalog["raw_columns_resolved"].astype(str).str.len().eq(0)]
print("features_sem_origem_bruta_resolvida =", missing_raw["feature"].tolist())


In [ ]:
driver_catalog = pd.DataFrame(
    [
        {
            "driver": name,
            "nome_interpretavel": doc.title,
            "flag_feature_real": doc.flag_feature,
            "definicao": doc.definition,
            "tabela_onde_nasce": doc.born_table,
            "colunas_que_alimentam": " | ".join(doc.source_columns),
            "fontes_brutas": " | ".join(doc.raw_sources),
            "regra_logica": doc.rule,
            "interpretacao": doc.interpretation,
            "limitacoes": " | ".join(doc.limitations),
            "hipoteses": " | ".join(doc.hypotheses),
            "graficos": " | ".join(doc.charts),
            "csvs_finais": " | ".join(doc.final_csvs),
        }
        for name, doc in DRIVER_DOCUMENTATION.items()
    ]
)

show_scroll_df("Catálogo de drivers das semanas-problema", driver_catalog, max_height=680)

md(
    '''
    #### Como ler exemplos importantes

    - `solo_exposto` nasce da flag `high_soil_flag`, que usa `soil_pct_week`, derivado de `b1_pct_solo` no `ndvi_metadata.csv`.
    - `falha_de_dose_na_adubacao` nasce da flag `fert_risk_flag`, que compara dose aplicada e dose configurada nas camadas EKOS de adubação.
    - `risco_de_motor` nasce da flag `engine_risk_flag`, alimentada por temperatura do motor, rotação do motor e consumo de combustível.
    - `pressao_de_pragas` nasce da flag `pest_risk_flag`, alimentada por contagens e eventos do MIIP/EKOS Pest.
    - `estresse_climatico` nasce da flag `weather_stress_flag`, alimentada por chuva, evapotranspiração e balanço hídrico.
    '''
)

if "driver" in event_driver_lift.columns:
    driver_evidence = event_driver_lift.merge(driver_catalog, on="driver", how="left")
    show_scroll_df(
        "Evidência real dos drivers em event_driver_lift.csv",
        driver_evidence,
        max_height=620,
    )


In [ ]:
all_output_frames = {
    path.name: pd.read_csv(path)
    for path in sorted(OUTPUTS_DIR.glob("*.csv"))
}

csv_catalog = pd.DataFrame(
    [
        {
            "csv": name,
            "linhas": len(frame),
            "colunas": len(frame.columns),
            "funcao_geradora": (
                f"{CSV_REGISTRY[name].module}.{CSV_REGISTRY[name].function}"
                if name in CSV_REGISTRY
                else "csv_exportado_pelo_fluxo_completo"
            ),
            "dependencias": (
                " | ".join(CSV_REGISTRY[name].dependencies)
                if name in CSV_REGISTRY
                else "workspace/output exportado"
            ),
            "hipoteses": (
                " | ".join(CSV_REGISTRY[name].related_hypotheses)
                if name in CSV_REGISTRY
                else ""
            ),
            "graficos": (
                " | ".join(CSV_REGISTRY[name].related_charts)
                if name in CSV_REGISTRY
                else ""
            ),
            "descricao": (
                CSV_REGISTRY[name].description
                if name in CSV_REGISTRY
                else "CSV real exportado pelo pipeline completo; documentação coluna-a-coluna inferida abaixo."
            ),
        }
        for name, frame in all_output_frames.items()
    ]
)

show_scroll_df("Catálogo dos CSVs gerados em notebook_outputs/complete_ndvi", csv_catalog, max_height=560)

workspace_column_catalog = build_workspace_column_catalog(workspace, all_output_frames)
workspace_column_catalog = workspace_column_catalog.sort_values(["kind", "table", "column"]).reset_index(drop=True)
show_scroll_df(
    "Dicionário de colunas intermediárias e finais geradas",
    workspace_column_catalog[
        [
            "kind",
            "table",
            "column",
            "dtype",
            "created_here",
            "documentation",
            "usage_status",
            "examples",
        ]
    ],
    max_height=760,
)

print("total_colunas_documentadas_no_notebook =", len(workspace_column_catalog))
print("csvs_documentados_no_notebook =", len(csv_catalog))


In [ ]:
ndvi_band_policy = pd.DataFrame(
    [
        {
            "campo": "b1_*",
            "papel": "banda NDVI usada no pipeline",
            "uso": "b1_mean, b1_std, b1_pct_solo, b1_pct_veg_densa e b1_valid_pixels alimentam features.",
            "motivo": "No pacote local, b1 representa a banda analítica de NDVI.",
        },
        {
            "campo": "b2_* / b2_valid_pixels",
            "papel": "banda/máscara auxiliar documentada",
            "uso": "não entra no cálculo principal de NDVI",
            "motivo": "Não representa a métrica principal de vigor usada no projeto; fica disponível para auditoria.",
        },
        {
            "campo": "b3_* / b3_valid_pixels",
            "papel": "banda/máscara auxiliar documentada",
            "uso": "não entra no cálculo principal de NDVI",
            "motivo": "Não representa a métrica principal de vigor usada no projeto; fica disponível para auditoria.",
        },
        {
            "campo": "b1_valid_pixels",
            "papel": "filtro de qualidade",
            "uso": "linhas com b1_valid_pixels <= 0 são removidas antes da análise",
            "motivo": "Sem pixel válido, a cena não tem NDVI utilizável.",
        },
    ]
)
show_scroll_df("Política de uso das bandas b1, b2 e b3", ndvi_band_policy, max_height=360)



## 3.1 `ndvi_clean.csv`

### O que é
É a **primeira base NDVI realmente analítica** do projeto. Ela pega o bruto do OneSoil e transforma em algo utilizável para comparação temporal.

### Entradas
- bruto `ndvi_metadata.csv`;
- `area_metadata` (mapeamento de `season_id` para grão/silagem e 4.0/convencional);
- `weather_daily` (apenas para marcar se existe cobertura climática naquele período).

### Filtros principais
- converte `date`;
- **remove linhas com `b1_valid_pixels <= 0`**;
- ordena por `season_id` e `date`.

### Colunas derivadas importantes
- `week_start`;
- `ndvi_mean = b1_mean`;
- `ndvi_std = b1_std`;
- `soil_pct = b1_pct_solo`;
- `dense_veg_pct = b1_pct_veg_densa`;
- `ndvi_delta`;
- `ndvi_auc`;
- `has_weather_coverage`.

### Onde é usado depois
- `ndvi_stats_by_area.csv`
- `ndvi_outliers.csv`
- `pairwise_weekly_features` (interno)
- indiretamente todo o restante da análise

### Hipóteses / gráficos dependentes
- todos os gráficos de NDVI derivam dele, direta ou indiretamente.


In [ ]:

show_function('pairwise.py', 'build_ndvi_clean')
show_function('pairwise.py', '_week_start')
show_function('pairwise.py', '_cumulative_auc')


In [ ]:

transform_map_ndvi_clean = pd.DataFrame([
    ['date', 'date', 'normalização da data'],
    ['b1_valid_pixels', 'filtro de entrada', 'somente linhas > 0 entram'],
    ['b1_mean', 'ndvi_mean', 'NDVI médio da cena'],
    ['b1_std', 'ndvi_std', 'dispersão do NDVI'],
    ['b1_pct_solo', 'soil_pct', 'percentual de solo exposto'],
    ['b1_pct_veg_densa', 'dense_veg_pct', 'percentual de vegetação densa'],
    ['date', 'week_start', 'âncora semanal'],
    ['ndvi_mean', 'ndvi_delta', 'variação entre cenas consecutivas'],
    ['ndvi_mean + date', 'ndvi_auc', 'área acumulada sob a curva'],
])
show_df('Mapa de transformação do bruto para ndvi_clean', transform_map_ndvi_clean, 20)
show_df('Amostra de ndvi_clean.csv', ndvi_clean, 8)


In [ ]:

# reprodução simplificada de parte do ndvi_clean usando o bruto
area_metadata_demo = area_inventory[['season_id','area_label','treatment','crop_type','comparison_pair']].drop_duplicates().copy()
raw_demo = ndvi_raw.copy()
raw_demo['date'] = pd.to_datetime(raw_demo['filename'].str.extract(r'(\d{4}-\d{2}-\d{2})')[0], errors='coerce')
raw_demo = raw_demo[raw_demo['b1_valid_pixels'].fillna(0) > 0].copy()
raw_demo = raw_demo.merge(area_metadata_demo, on='season_id', how='left')
raw_demo['week_start'] = pd.to_datetime(raw_demo['date']).dt.to_period('W').dt.start_time
raw_demo['ndvi_mean'] = raw_demo['b1_mean']
raw_demo['ndvi_std'] = raw_demo['b1_std']
raw_demo['soil_pct'] = raw_demo['b1_pct_solo']
raw_demo['dense_veg_pct'] = raw_demo['b1_pct_veg_densa']
show_df('Reprodução simplificada da lógica central de ndvi_clean (demo)', raw_demo[[
    'filename','season_id','date','week_start','ndvi_mean','ndvi_std','soil_pct','dense_veg_pct','area_label','treatment'
]], 8)



## 3.2 `weather_daily.csv`

### O que é
É a base climática diária derivada da série horária da estação Metos.

### Entradas
- CSV horário da Metos.

### Filtros / regras
- agregação diária por `date`;
- soma chuva e evapotranspiração;
- média de temperatura, umidade, radiação e vento;
- calcula `water_balance_mm`;
- cria médias móveis (`ma7` e `ma14`) e `week_start`.

### Onde é usado depois
- `ndvi_clean` (marcação de cobertura climática);
- `weather_weekly` (interno);
- `pairwise_weekly_features` (interno).


In [ ]:

show_function('pairwise.py', 'build_weather_daily')
show_function('pairwise.py', 'build_weather_weekly')
show_df('Amostra de weather_daily.csv', weather_daily, 8)



## 3.3 `ops_area_daily.csv`

### O que é
É a consolidação diária das camadas operacionais do EKOS por área.

### Entradas
- plantio;
- colheita;
- adubação;
- pulverização;
- sobreposição;
- velocidade;
- estado de máquina;
- paradas.

### Lógica principal
- cada camada é lida;
- as geometrias são atribuídas a `season_id`;
- cada layer é resumido por `date` e `season_id`;
- no fim, tudo é unido por área/dia.

### Papel no foco NDVI
Não é eixo principal. Entra como **explicador** para semanas ruins e diferenças entre os pares.


In [ ]:

show_function('pairwise.py', 'build_ops_area_daily')
show_df('Amostra de ops_area_daily.csv', ops_area_daily, 8)



## 3.4 `miip_daily.csv`

### O que é
É a consolidação diária de pragas/armadilhas por área.

### Entradas
- `traps_data.csv`
- `traps_list.csv`
- `pest_list.csv`
- `pest_details.csv`
- `traps_events.csv` quando disponível

### Papel no foco NDVI
Serve para identificar se **pressão de pragas** aparece mais nas semanas-problema do NDVI.


In [ ]:

show_function('pairwise.py', 'build_miip_daily')
show_df('Amostra de miip_daily.csv', miip_daily, 8)



## 3.5 `pairwise_weekly_features` (tabela interna importante)

### O que é
É a base semanal que junta:
- NDVI semanal;
- clima semanal;
- operação semanal;
- MIIP semanal;
- metadados da área.

### Observação importante
Ela **é central na lógica**, mas não vem exportada como CSV final neste pacote refrescado. Mesmo assim, sem ela, a `ndvi_phase_timeline` não existiria.


In [ ]:

show_function('pairwise.py', 'build_pairwise_weekly_features')
show_function('pairwise.py', 'build_ndvi_weekly')



## 3.6 `ndvi_phase_timeline.csv`

### O que é
É a **timeline semanal enriquecida**, a tabela mais importante para entender queda, recuperação, baixo vigor, fase do ciclo e drivers.

### Entradas
- `pairwise_weekly_features`;
- `ops_support_weekly` (alarmes, telemetria, motor, combustível etc.).

### Colunas-chave adicionadas
- `ndvi_norm_week`
- `peak_week_start`
- `weeks_from_peak`
- flags de risco (`high_soil_flag`, `pest_risk_flag`, `ops_risk_flag`, etc.)
- `phase`
- `event_type`
- `primary_driver`
- `story_sentence`

### Onde é usado depois
- `pair_weekly_gaps.csv`
- `pair_effect_tests.csv`
- `event_driver_lift.csv`
- `final_hypothesis_register.csv`
- `decision_summary.csv`


In [ ]:

show_function('ndvi_deepdive.py', 'build_ndvi_phase_timeline')
show_df('Amostra de ndvi_phase_timeline.csv', ndvi_phase_timeline, 8)



## 3.7 `ndvi_stats_by_area.csv`

### O que é
É a estatística descritiva do NDVI por área.

### Entradas
- `ndvi_clean`.

### O que resume
- número de imagens válidas;
- semanas com cenas;
- início e fim das cenas;
- média, mediana, desvio, quartis, CV, assimetria e curtose.

### Papel
Resume o comportamento global do NDVI de cada área.


In [ ]:

show_function('complete_analysis.py', 'build_ndvi_stats_by_area')
show_df('Amostra de ndvi_stats_by_area.csv', ndvi_stats_by_area, 8)



## 3.8 `pair_weekly_gaps.csv`

### O que é
É a tabela que coloca, na mesma semana e no mesmo par, o **4.0** ao lado do **convencional** e calcula os gaps `4.0 - convencional`.

### Papel
É a base mais intuitiva para responder:
- em quais semanas o 4.0 ficou acima?
- em quais ficou abaixo?
- o gap foi grande ou pequeno?


In [ ]:

show_function('complete_analysis.py', 'build_pair_weekly_gaps')
show_df('Amostra de pair_weekly_gaps.csv', pair_weekly_gaps, 8)


## 3.9 `ndvi_outliers.csv`

### O que é
É a tabela que identifica **cenas de NDVI atípicas dentro da série de cada área**.

### Como foi feito
A função `build_ndvi_outliers` trabalha **área por área (`season_id`)**. Para cada série, ela calcula:
- **média** e **desvio-padrão** do `ndvi_mean`;
- **mediana** e **MAD** (desvio absoluto mediano).

Depois ela gera duas medidas:
- `ndvi_zscore = (valor - média) / desvio-padrão`
- `ndvi_robust_zscore = 0.6745 * (valor - mediana) / MAD`

### Critério usado no projeto
A cena é marcada como outlier quando atende **pelo menos uma** das condições:
- `|ndvi_zscore| >= 2.0`
- `|ndvi_robust_zscore| >= 3.5`

### Por que usar duas regras
- o **z-score clássico** funciona bem quando a série está razoavelmente estável;
- o **robust z-score** protege melhor contra séries que já têm valores extremos.

### Papel analítico
Separar eventos realmente atípicos da oscilação normal da série. Isso ajuda a não confundir um ponto extremo com uma tendência estrutural de perda ou ganho de vigor.


In [ ]:

show_function('complete_analysis.py', 'build_ndvi_outliers')
show_df('Amostra de ndvi_outliers.csv', ndvi_outliers, 8)


In [ ]:

# reprodução resumida do raciocínio de outlier
out_demo = ndvi_clean[['season_id','date','area_label','comparison_pair','ndvi_mean','soil_pct','dense_veg_pct']].copy()
out_demo['ndvi_zscore_demo'] = out_demo.groupby('season_id')['ndvi_mean'].transform(lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) not in [0, np.nan] else np.nan)
med = out_demo.groupby('season_id')['ndvi_mean'].transform('median')
mad = out_demo.groupby('season_id')['ndvi_mean'].transform(lambda s: np.median(np.abs(s - np.median(s))))
out_demo['ndvi_robust_zscore_demo'] = 0.6745 * (out_demo['ndvi_mean'] - med) / mad.replace(0, np.nan)
out_demo['outlier_demo'] = out_demo['ndvi_zscore_demo'].abs().gt(3) | out_demo['ndvi_robust_zscore_demo'].abs().gt(3.5)
show_df('Demonstração da lógica de outliers a partir de ndvi_clean', out_demo, 8)
print('Outliers (demo) =', int(out_demo['outlier_demo'].fillna(False).sum()))
print('Outliers (csv final) =', int(ndvi_outliers['outlier_flag'].fillna(False).sum()))



## 3.10 `pair_effect_tests.csv`

### O que é
É a tabela **principal** de efeito pareado do projeto.

### O que ela testa
Para cada par (`grao` e `silagem`) e para cada métrica relevante, a função compara semanalmente 4.0 versus convencional.

### Saídas importantes
- `advantage_4_0`
- `ci_low`, `ci_high`
- `p_value`
- `paired_effect_size`
- `winner`
- `evidence_level`

### Interpretação
- `advantage_4_0 > 0` favorece 4.0
- `advantage_4_0 < 0` favorece convencional
- `p_value` mede compatibilidade com ausência de diferença
- `IC95%` mostra a faixa plausível do efeito


In [ ]:

show_function('ndvi_crispdm.py', 'build_pair_effect_tests')
show_function('ndvi_crispdm.py', '_pair_effect_row')
show_function('ndvi_crispdm.py', '_bootstrap_mean_ci')
show_function('ndvi_crispdm.py', '_sign_flip_p_value')
show_function('ndvi_crispdm.py', '_paired_effect_size')
show_function('ndvi_crispdm.py', '_effect_evidence_level')
show_df('Amostra de pair_effect_tests.csv', pair_effect_tests, 14)


## 3.11 `pair_classic_tests.csv`

### O que é
É a validação estatística clássica dos gaps semanais entre 4.0 e convencional.

### Como funciona
1. calcula o vetor de gaps semanais da métrica;
2. aplica **Shapiro-Wilk** para verificar se os gaps parecem compatíveis com normalidade;
3. se a normalidade parecer aceitável, usa **t-test de uma amostra contra zero**;
4. se a normalidade não parecer boa, usa **Wilcoxon**;
5. grava `recommended_test` e `recommended_p_value`.

### Hipótese nula
Em linguagem simples: **o gap médio ou central é zero**, ou seja, não existe diferença sistemática entre 4.0 e convencional naquela métrica.

### Como interpretar
- `recommended_p_value` pequeno → há menos compatibilidade com a hipótese de “sem diferença”;
- `paired_effect_size_dz` ajuda a dizer se a diferença parece pequena ou mais relevante;
- `significant_0_05` é apenas um marcador prático para o corte de 0,05.


In [ ]:

show_function('complete_analysis.py', 'build_pair_classic_tests')
show_function('complete_analysis.py', '_shapiro_pvalue')
show_function('complete_analysis.py', '_ttest_1samp_pvalue')
show_function('complete_analysis.py', '_wilcoxon_pvalue')
show_df('Amostra de pair_classic_tests.csv', pair_classic_tests, 14)



## 3.12 `weekly_correlations.csv`

### O que é
É a tabela de correlação entre o alvo semanal de NDVI e os possíveis explicadores.

### Como foi feita
- correlação de Pearson;
- correlação de Spearman;
- classificação de direção e força.

### Hipótese nula
A correlação é zero.


In [ ]:

show_function('complete_analysis.py', 'build_weekly_correlations')
show_df('Amostra de weekly_correlations.csv', weekly_correlations, 14)


## 3.13 `event_driver_lift.csv`

### O que é
Mostra quais drivers aparecem **mais do que o normal** nas semanas classificadas como problemáticas para o NDVI.

### Como o projeto define “semana-problema”
A função `build_event_driver_lift` marca uma semana como problemática quando ocorre pelo menos uma destas situações:
- `major_drop_flag = True`;
- `low_vigor_flag = True`;
- `event_type` igual a `queda`, `queda_forte` ou `baixo_vigor`.

### Como o cálculo é feito
Para cada driver e para cada par:
- `problem_rate` = proporção de semanas-problema em que o driver apareceu;
- `baseline_rate` = proporção de semanas não problemáticas em que o driver apareceu;
- `delta_pp = (problem_rate - baseline_rate) * 100`;
- `lift_ratio = problem_rate / baseline_rate` quando a taxa de baseline é positiva.

### Como interpretar
- `delta_pp` alto → o driver ficou muito mais frequente nas semanas ruins;
- `lift_ratio` alto → a presença do driver cresceu bastante em relação ao normal;
- `evidence_level` resume a força do sinal observado.

### Papel analítico
Essa tabela não prova causalidade sozinha, mas ajuda a identificar **quais sinais ficaram sobre-representados justamente quando o NDVI piorou**.


In [ ]:

show_function('ndvi_crispdm.py', 'build_event_driver_lift')
show_df('Amostra de event_driver_lift.csv', event_driver_lift, 14)



## 3.14 `final_hypothesis_register.csv` e `decision_summary.csv`

### O que são
São as duas tabelas finais de fechamento da narrativa.

- `final_hypothesis_register.csv` → status formal de H1, H2, H3 e H4;
- `decision_summary.csv` → resumo por par com mensagem executiva.

### Importante
A hipótese **não é definida só por p-value**. O projeto combina:
- efeito observado;
- evidência estatística;
- drivers;
- outlook;
- limites conhecidos.


In [ ]:

show_function('ndvi_crispdm.py', 'build_final_hypothesis_register')
show_function('ndvi_crispdm.py', 'build_decision_summary')
show_df('final_hypothesis_register.csv', final_hypothesis_register, 20)
show_df('decision_summary.csv', decision_summary, 10)



# 4. Filtros completos do pipeline

Abaixo está um catálogo consolidado dos filtros e regras que realmente mudam a base analítica.


In [ ]:

filters_catalog = pd.DataFrame([
    ['NDVI', 'build_ndvi_clean', 'remove cenas sem NDVI utilizável', 'b1_valid_pixels > 0'],
    ['Tempo', 'build_ndvi_clean / build_weather_daily', 'normaliza datas e cria week_start', 'datas convertidas e agregadas por semana'],
    ['Mapeamento de áreas', 'build_area_metadata', 'transforma season_id em linguagem de negócio', 'OFFICIAL_AREA_MAPPING'],
    ['Clima', 'build_weather_daily / build_weather_weekly', 'agrega série horária para diária e semanal', 'sum/mean/min/max por dia e semana'],
    ['Operação', 'build_ops_area_daily', 'atribui geometrias às áreas e resume por dia', 'groupby season_id + date'],
    ['MIIP', 'build_miip_daily', 'resume leituras e eventos de pragas por dia', 'groupby season_id + date'],
    ['Integração semanal', 'build_pairwise_weekly_features', 'junta NDVI, clima, operação e MIIP', 'join por season_id + week_start'],
    ['Normalização temporal', 'build_ndvi_phase_timeline', 'normaliza NDVI por área', 'ndvi_norm_week entre 0 e 1'],
    ['Eventos', 'build_ndvi_phase_timeline', 'identifica queda, recuperação, pico, baixo vigor', 'limiares em ndvi_delta_week e ndvi_norm_week'],
    ['Outliers', 'build_ndvi_outliers', 'marca cenas atípicas', 'z-score e robust z-score'],
    ['Evidência estatística', 'build_pair_effect_tests / build_pair_classic_tests', 'mede força da diferença', 'p-value, IC95%, effect size'],
    ['Hipóteses', 'build_final_hypothesis_register', 'fecha H1-H4', 'combina testes, drivers e outlook'],
])
show_df('Catálogo de filtros e regras relevantes', filters_catalog, 30)


# 5. Estatística usada no projeto

Esta seção explica, em linguagem direta, as técnicas estatísticas que aparecem no notebook e como elas entram nas decisões do projeto. A ideia aqui não é só mostrar o número final, mas explicar o **que ele mede**, **qual pergunta ele responde** e **como deve ser interpretado**.


In [ ]:
stats_explained = pd.DataFrame([
    ['pair_effect_tests', 'teste principal de efeito pareado com IC e p-value por randomização', 'não há vantagem sistemática do 4.0', 'há vantagem sistemática em uma direção', 'compara 4.0 vs convencional por semana nas métricas principais', 'advantage_4_0, IC95%, p_value, effect_size'],
    ['pair_classic_tests / Shapiro', 'teste de normalidade', 'os gaps são compatíveis com normalidade', 'os gaps não são compatíveis com normalidade', 'ajuda a decidir entre t-test e Wilcoxon', 'normality_shapiro_p'],
    ['pair_classic_tests / t-test', 'teste t de uma amostra vs zero', 'a média do gap é zero', 'a média do gap não é zero', 'usado quando a distribuição do gap parece normal', 'ttest_gap_vs_zero_p'],
    ['pair_classic_tests / Wilcoxon', 'teste não paramétrico', 'o centro das diferenças é zero', 'o centro das diferenças não é zero', 'usado quando a normalidade não parece adequada', 'wilcoxon_gap_vs_zero_p'],
    ['ndvi_outliers', 'z-score e robust z-score', 'a cena está dentro do padrão da área', 'a cena é atípica para a própria área', 'detecta cenas extremas na série NDVI', 'outlier_flag, ndvi_zscore, ndvi_robust_zscore'],
    ['weekly_correlations', 'Pearson e Spearman', 'a correlação é zero', 'a correlação é diferente de zero', 'mede associação entre NDVI e variáveis explicativas', 'pearson_p, spearman_p'],
    ['event_driver_lift', 'diferença de taxas em semanas-problema', 'o driver aparece na mesma taxa das semanas normais', 'o driver aparece mais nas semanas-problema', 'mede sobre-representação de drivers', 'problem_rate, baseline_rate, delta_pp, lift_ratio'],
])
show_df('Explicação da estatística usada no projeto', stats_explained, 20)


### Conceitos estatísticos importantes

**p.p. (pontos percentuais)**  
Quando o notebook mostra `delta_pp`, ele está medindo a diferença entre duas taxas em **pontos percentuais**, não em porcentagem relativa.  
Exemplo: se um driver aparece em 60% das semanas-problema e em 25% das semanas normais, então `delta_pp = 60 - 25 = 35 p.p.`.

**Hipótese nula (H0)**  
É a hipótese de referência. Em geral, ela representa a ideia de **ausência de diferença** ou **ausência de relação**.  
Exemplos no projeto:
- nos testes de efeito, H0 significa “não existe vantagem sistemática entre 4.0 e convencional”;
- nas correlações, H0 significa “a correlação é zero”;
- na análise de tendência, H0 significa “a inclinação é zero”.

**Hipótese alternativa (H1)**  
É a hipótese concorrente à hipótese nula. Em geral, representa a ideia de que **há diferença**, **há relação** ou **há tendência**.

**p-value**  
É uma medida de compatibilidade entre os dados observados e a hipótese nula.  
Ele responde à pergunta:  
> “Se a hipótese nula fosse verdadeira, quão estranho seria observar um resultado pelo menos tão extremo quanto este?”

Interpretação prática:
- **p-value pequeno** → o resultado fica menos compatível com H0;
- **p-value grande** → o resultado ainda é compatível com H0.

Importante: o p-value **não mede a chance de H0 ser verdadeira** e **não prova causa** sozinho.

**IC95% (intervalo de confiança de 95%)**  
É um intervalo de valores plausíveis para o efeito observado. No projeto, ele ajuda a qualificar a estabilidade da diferença entre 4.0 e convencional.  
Quando o intervalo fica todo de um lado do zero, isso reforça a leitura de que há um efeito consistente naquela direção.

**Tamanho de efeito**  
Mesmo quando o p-value é pequeno, o projeto também olha para a **magnitude do efeito**. Isso evita tratar diferenças muito pequenas como se fossem automaticamente relevantes.

**Shapiro-Wilk**  
É um teste de normalidade. Ele ajuda o projeto a decidir se os gaps semanais parecem suficientemente próximos de uma distribuição normal.

**t-test de uma amostra contra zero**  
É usado quando os gaps semanais parecem compatíveis com normalidade.  
Hipótese nula: a média do gap é zero.

**Wilcoxon**  
É o teste alternativo quando a normalidade não parece boa.  
Hipótese nula: o centro das diferenças está em zero.  
Ele é útil quando se quer uma comparação menos dependente da suposição de normalidade.

**z-score**  
Mede quantos desvios-padrão um valor está acima ou abaixo da média.  
No notebook, ele ajuda a identificar cenas de NDVI muito fora do padrão da própria área.

**robust z-score**  
É uma versão mais robusta do z-score, baseada em mediana e MAD (desvio absoluto mediano).  
Ela é útil quando a série já tem valores extremos e você quer uma medida menos sensível a eles.

**Pearson e Spearman**  
- **Pearson** mede associação linear;
- **Spearman** mede associação monotônica por postos.  
O notebook usa os dois para não depender de uma única noção de correlação.

### Como essa estatística entra na decisão final

O projeto **não decide as hipóteses só por p-value**.  
A decisão final combina:
- direção do efeito;
- tamanho do efeito;
- IC95%;
- consistência temporal;
- drivers sobre-representados;
- outlook do par;
- e limites conhecidos do pacote de dados.

# 6. Gráfico por gráfico

A partir daqui, cada gráfico aparece ligado à sua origem, ao cálculo que o gerou e à pergunta que ele ajuda a responder.



## 6.1 Evolução semanal do NDVI por área

- **DataFrame de origem:** `ndvi_phase_timeline`
- **Função que o alimenta:** `build_ndvi_phase_timeline`
- **Hipótese relacionada:** H1
- **Pergunta respondida:** quem sustentou maior nível temporal de NDVI?


In [ ]:

plot_df = ndvi_phase_timeline.copy()
plot_df['week_start'] = pd.to_datetime(plot_df['week_start'])
for pair, sub in plot_df.groupby('comparison_pair'):
    plt.figure(figsize=(12, 4))
    for area, area_df in sub.groupby('area_label'):
        plt.plot(area_df['week_start'], area_df['ndvi_mean_week'], marker='o', label=area)
    plt.title(f'NDVI médio semanal por área — par {pair}')
    plt.xlabel('Semana')
    plt.ylabel('NDVI médio semanal')
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()



**Leitura:** este gráfico é o coração da hipótese H1. Quando a linha do 4.0 fica acima de forma consistente, isso favorece H1; quando a do convencional fica acima, H1 perde força naquele par.



## 6.2 Gap semanal de NDVI: 4.0 − convencional

- **DataFrame de origem:** `pair_weekly_gaps`
- **Função geradora:** `build_pair_weekly_gaps`
- **Hipótese relacionada:** H1 e H2
- **Pergunta respondida:** em quais semanas o 4.0 ficou acima ou abaixo?


In [ ]:

for pair, sub in pair_weekly_gaps.groupby('comparison_pair'):
    x = pd.to_datetime(sub['week_start'])
    y = sub['gap_ndvi_mean_week_4_0_minus_convencional']
    plt.figure(figsize=(12, 4))
    plt.axhline(0, color='black', linewidth=1)
    plt.plot(x, y, marker='o')
    plt.title(f'Gap semanal de NDVI médio — 4.0 menos convencional ({pair})')
    plt.xlabel('Semana')
    plt.ylabel('Gap de NDVI')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()



**Leitura:** valores acima de zero favorecem o 4.0; valores abaixo de zero favorecem o convencional. É uma das visualizações mais diretas do efeito temporal.



## 6.3 NDVI médio consolidado por área

- **DataFrame de origem:** `ndvi_stats_by_area`
- **Função geradora:** `build_ndvi_stats_by_area`
- **Hipótese relacionada:** H1


In [ ]:

plt.figure(figsize=(10,4))
plt.bar(ndvi_stats_by_area['area_label'], ndvi_stats_by_area['mean'])
plt.title('NDVI médio consolidado por área')
plt.ylabel('NDVI médio')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()



**Leitura:** resume o nível médio global do NDVI por área. Não substitui a leitura temporal, mas ajuda a visualizar o panorama geral.


## 6.4 Outliers de NDVI

- **DataFrame de origem:** `ndvi_outliers`
- **Função geradora:** `build_ndvi_outliers`
- **Pergunta respondida:** quais cenas ficaram muito fora do padrão da própria área?

### Como o gráfico deve ser lido
Cada ponto representa uma cena NDVI. Os pontos maiores são os que receberam `outlier_flag = True`.  
A classificação foi feita comparando cada valor com o comportamento histórico da própria área, e não comparando áreas diferentes entre si.

### Critério usado
Uma cena é marcada como atípica quando:
- `|ndvi_zscore| >= 2.0`, **ou**
- `|ndvi_robust_zscore| >= 3.5`.

Isso significa que o projeto usa duas lentes ao mesmo tempo: uma baseada em média/desvio-padrão e outra baseada em mediana/MAD.


In [ ]:
out_plot = ndvi_outliers.copy()
out_plot['date'] = pd.to_datetime(out_plot['date'])

plt.figure(figsize=(12,4))
for area, sub in out_plot.groupby('area_label'):
    plt.scatter(
        sub['date'],
        sub['ndvi_mean'],
        s=np.where(sub['outlier_flag'].fillna(False), 90, 22),
        alpha=np.where(sub['outlier_flag'].fillna(False), 0.95, 0.55),
        label=area
    )
plt.title('Cenas de NDVI por área com destaque para pontos atípicos')
plt.xlabel('Data')
plt.ylabel('NDVI médio')
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

outlier_summary = (
    ndvi_outliers.assign(
        outlier_flag=ndvi_outliers['outlier_flag'].fillna(False),
        outlier_zscore_flag=ndvi_outliers['outlier_zscore_flag'].fillna(False),
        outlier_robust_flag=ndvi_outliers['outlier_robust_flag'].fillna(False),
    )
    .groupby('area_label', as_index=False)
    .agg(
        cenas=('date', 'count'),
        outliers=('outlier_flag', 'sum'),
        por_zscore=('outlier_zscore_flag', 'sum'),
        por_robust_zscore=('outlier_robust_flag', 'sum'),
        menor_zscore=('ndvi_zscore', 'min'),
        maior_zscore=('ndvi_zscore', 'max'),
    )
)
show_df('Resumo dos outliers por área', outlier_summary, 20)

show_df(
    'Exemplos de cenas classificadas como outlier',
    ndvi_outliers[ndvi_outliers['outlier_flag'].fillna(False)][[
        'area_label','date','ndvi_mean','ndvi_zscore','ndvi_robust_zscore','outlier_direction'
    ]].sort_values(['area_label','date']),
    20,
)



**Leitura:** pontos destacados indicam cenas que ficaram fora do comportamento esperado da série daquela área.


## 6.5 Drivers das semanas-problema

- **DataFrame de origem:** `event_driver_lift`
- **Função geradora:** `build_event_driver_lift`
- **Hipótese relacionada:** H3

### Como o gráfico foi construído
O projeto compara, para cada driver, duas frequências:
- a frequência nas **semanas-problema** (`problem_rate`);
- a frequência nas **semanas normais** (`baseline_rate`).

A barra mostra `delta_pp`, isto é, a diferença entre essas duas taxas em **pontos percentuais**.

### Como interpretar
- barra alta e positiva → o driver apareceu bem mais nas semanas ruins do que nas semanas normais;
- barra perto de zero → o driver não mudou muito de comportamento;
- `lift_ratio` complementa a leitura mostrando a razão entre as duas taxas.

### O que esse gráfico não faz
Ele **não prova causalidade sozinho**. O que ele mostra é sobre-representação: o driver ficou mais presente justamente nas semanas em que o NDVI caiu ou entrou em baixo vigor.


In [ ]:
driver_plot = event_driver_lift.copy()
driver_plot['problem_rate_pct'] = driver_plot['problem_rate'] * 100
driver_plot['baseline_rate_pct'] = driver_plot['baseline_rate'] * 100

for pair, sub in driver_plot.groupby('comparison_pair'):
    sub = sub.sort_values('delta_pp', ascending=True)
    plt.figure(figsize=(10,5))
    plt.barh(sub['driver'], sub['delta_pp'])
    plt.title(f'Drivers mais associados às semanas-problema — {pair}')
    plt.xlabel('Diferença em pontos percentuais (problem_rate - baseline_rate)')
    plt.ylabel('Driver')
    plt.tight_layout()
    plt.show()

    show_df(
        f'Tabela de apoio dos drivers — {pair}',
        sub[['driver','problem_weeks','problem_rate_pct','baseline_rate_pct','delta_pp','lift_ratio','evidence_level']]
        .rename(columns={
            'problem_rate_pct':'problem_rate_%',
            'baseline_rate_pct':'baseline_rate_%'
        })
        .sort_values('delta_pp', ascending=False),
        20
    )



**Leitura:** esse gráfico mostra o que apareceu mais do que o normal nas semanas em que o NDVI ficou problemático.


## 6.6 Resumo visual das hipóteses H1–H4

- **DataFrame de origem:** `final_hypothesis_register`
- **Função geradora:** `build_final_hypothesis_register`
- **Pergunta respondida:** o que foi suportado, o que não foi suportado e o que ainda ficou inconclusivo?

Em vez de um painel de calor, esta versão usa uma tabela mais legível, porque a leitura textual do status ajuda mais na apresentação.


In [ ]:
hyp_table = final_hypothesis_register.copy()
hyp_table['comparison_pair'] = hyp_table['comparison_pair'].str.title()
hyp_table['status'] = hyp_table['status'].str.replace('nao', 'não', regex=False)

pivot_status = hyp_table.pivot(index='comparison_pair', columns='hypothesis_id', values='status')
pivot_basis = hyp_table.pivot(index='comparison_pair', columns='hypothesis_id', values='proof_basis')

def color_status(val):
    val = str(val).lower()
    if 'suportad' in val and 'não' not in val and 'nao' not in val:
        return 'background-color: #d9f2d9'
    if 'inconclus' in val:
        return 'background-color: #fff4cc'
    if 'não suportad' in val or 'nao suportad' in val:
        return 'background-color: #f8d7da'
    return ''

display(
    pivot_status.style
    .applymap(color_status)
    .set_caption('Status das hipóteses por par')
)

for pair in pivot_basis.index:
    md(f"### Base resumida das hipóteses — {pair}")
    display(pivot_basis.loc[[pair]].T.rename(columns={pair: 'base_da_decisao'}))



**Leitura:** aqui você enxerga rapidamente o fechamento formal das hipóteses por par.


# 7. Leitura integrada dos resultados

Nesta etapa, os resultados são reunidos em uma explicação única, conectando os gráficos, os testes e os limites do pacote de dados.


In [ ]:

show_df('Resumo decisório final por par', decision_summary, 10)


## O que aconteceu no par grão

No par **grão**, o projeto encontrou um cenário em que o **convencional** ficou acima do 4.0 no comportamento temporal do NDVI. Isso aparece nos testes principais e nos gaps semanais. O driver mais associado às semanas-problema foi **solo exposto**, o que sugere que parte da perda de desempenho vegetativo está associada à condição de cobertura da área e não apenas à presença ou ausência de tecnologia.

## O que aconteceu no par silagem

No par **silagem**, o **4.0** apareceu acima do convencional no NDVI médio semanal e também na área sob a curva. Ao mesmo tempo, as semanas-problema desse par apontaram mais fortemente para **risco de motor**, mostrando que, mesmo onde o 4.0 foi melhor no vigor vegetativo, ainda existiam sinais operacionais relevantes.

## Por que o 4.0 não foi melhor no geral

Porque o projeto não mede “tecnologia” como um selo automático de melhor resultado. O que ele mede é o **resultado observado no campo**, e esse resultado depende da combinação entre:
- fase do ciclo;
- condição da cobertura do solo;
- clima;
- pressão de pragas;
- consistência operacional;
- telemetria e eventos de máquina.

Por isso, o 4.0 pode aparecer melhor em um par e não em outro.

## Por que houve queda e depois recuperação

Porque o projeto trabalha com uma timeline semanal e classifica eventos. Nem toda queda significa colapso da lavoura. Algumas quedas fazem parte da transição de fase; outras sinalizam baixo vigor, estresse ou desvio operacional. A recuperação aparece quando a série volta a subir depois dessas semanas mais fracas.

## O que já está sustentado pelos dados

- existe diferença temporal de NDVI entre os pares;
- essa diferença não foi uniforme entre grão e silagem;
- há drivers sobre-representados nas semanas-problema;
- a camada econômica ainda não está fechada com o mesmo nível de robustez.


# 8. Fechamento

A forma mais clara de entender o projeto é seguir esta ordem:

1. bruto (`ndvi_metadata.csv`, Metos, EKOS, MIIP, solo);
2. `ndvi_clean.csv`;
3. `ndvi_phase_timeline.csv`;
4. `pair_weekly_gaps.csv`, `ndvi_stats_by_area.csv`, `ndvi_outliers.csv`;
5. `pair_effect_tests.csv`, `pair_classic_tests.csv`, `weekly_correlations.csv`, `event_driver_lift.csv`;
6. `final_hypothesis_register.csv` e `decision_summary.csv`.

Seguindo essa ordem, fica mais fácil enxergar como o dado bruto vira evidência estatística e, por fim, narrativa analítica.
